# Anonymity and Cooperation in Social Norms

**Strand 2 — Repeated games with UCB.**

## Basic setup

- Contextual bandits allow playing a game (perform an action) on the basis of a *type* or context. They choose an action (cooperate or defect) on the basis of the type of their interlocutor.
- We generate a population of bandits, each with a type, and study setups that range from full anonymity (one type) to no anonymity (a type per agent).
- Agents are matched at random and play a one-shot game (Stag-Hunt, PD, Hawk-Dove); the choice rule (ε-greedy / softmax / UCB) is the variable of interest.

## Preliminary results

- For ε-greedy and softmax, the more types/flags/tribes, the less the agents cooperate. The chance of encountering agents of a given type decreases with the number of flags, so uncertainty is higher and the safest option (defect) wins.
- For UCB, agents always cooperate and get better performance. Being a *rational optimist* about others (UCB) promotes cooperation.

**Note on structure:** This notebook contains three successive iterations of `BanditEnvironment` / `ContextualBandit` / `GridWorld`. They're preserved in-notebook (rather than extracted to a shared module) because they're an experimental progression rather than a single canonical implementation.

## Iteration 1 — Bandit setup

### Imports

In [ ]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
from collections import defaultdict
import seaborn as sns

### Parameters and games

In [ ]:
# Parameters
n_flags = 5
action_space=['cooperate','defect']
num_contexts = n_flags
num_arms = len(action_space)
hawk_dove_game = {'cooperate': {'cooperate':2,'defect':0},'defect': {'cooperate': 4,'defect':-2 }}
stag_hunt_game = {'cooperate': {'cooperate':3,'defect':0},'defect': {'cooperate': 1,'defect':1 }}
prisoner_dilemma_game = {
    'cooperate': {'cooperate': 3, 'defect': 0},  # (C,C) = Reward (R), (C,D) = Sucker (S)
    'defect': {'cooperate': 4, 'defect': 1}   # (D,C) = Temptation (T), (D,D) = Punishment (P)
}

### Environment class (iteration 1)

In [ ]:
# ENVIRONMENT DEFINITION
# Simulating a contextual bandit environment
class BanditEnvironment:
    def __init__(self, num_arms, num_contexts,game = prisoner_dilemma_game,n_flags=n_flags):
        self.game = game
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        self.timestep = 0
        # the following is in order to keep the q table
        self.action_translation = {0:'cooperate',1:'defect'}

    def get_reward_single_game(self, p1_choice, p2_choice):
        self.timestep += 1
        p1_reward = self.game[p1_choice][p2_choice]
        p2_reward = self.game[p2_choice][p1_choice]
        return p1_reward, p2_reward

### Agent class (iteration 1)

In [ ]:
# AGENT DEFINITION
class ContextualBandit:
    def __init__(self, num_arms, num_contexts, explore_param, decay, min_explore,flag,choice_type='egreedy'):
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        # self.q_table = np.zeros((num_contexts, num_arms))
        self.q_table = np.full((num_contexts, num_arms), -1.0)
        self.arm_counts = np.ones((num_contexts, num_arms))
        self.flag = flag
        self.choice_type = choice_type
        self.decay = decay
        self.min_explore = min_explore
        self.explore_param = explore_param
        self.time = 0
        self.UCB_estimation = np.zeros((num_contexts, num_arms))
    def choice(self, context):
        self.time += 1  # Increment global time step

        if self.choice_type == 'egreedy':
            # ε-greedy strategy:
            # With probability ε (explore_param), choose a random arm (exploration)
            # With probability 1 - ε, choose the best known arm (exploitation)
            if np.random.rand() < self.explore_param:
                choice = np.random.randint(self.num_arms)
            else:
                q_best = np.max(self.q_table[context])
                # In case of ties, choose randomly among best arms
                choice = np.random.choice(np.where(self.q_table[context] == q_best)[0])

        elif self.choice_type == 'ucb':
            # Upper Confidence Bound (UCB) strategy:
            # UCB(a) = Q(a) + c * sqrt(log(t) / N(a))
            #   - Q(a): average reward for arm a
            #   - c: exploration parameter (explore_param)
            #   - t: current timestep
            #   - N(a): number of times arm a has been selected
            # We use 1e-5 to avoid division by zero before any arm is tried
            self.UCB_estimation[context] = self.q_table[context] + \
                            self.explore_param * np.sqrt(np.log(self.time + 1) / (self.arm_counts[context] + 1e-5))
            choice = np.argmax(self.UCB_estimation[context])  # Pick arm with highest UCB estimate

        elif self.choice_type == 'softmax':
            # Softmax (Boltzmann) strategy:
            # Probability of choosing arm a:
            #   P(a) = exp(Q(a)/T) / sum(exp(Q(k)/T)) for all arms k
            #   - T: temperature (explore_param), higher means more exploration
            q_values = self.q_table[context]
            # Subtracting max for numerical stability (avoids overflow in exp)
            exp_q = np.exp((q_values - np.max(q_values)) / self.explore_param)
            probs = exp_q / np.sum(exp_q)  # Normalize to get a probability distribution

            # Sample from the resulting softmax distribution
            choice = np.random.choice(self.num_arms, p=probs)

        self.arm_counts[context, choice] += 1
        return choice

    def print_report(self):
        print('This is the q_table:')
        print(self.q_table)
        print('This is the UCB estimation:')
        print(self.UCB_estimation)
        print('This is the arm counts:')
        print(self.arm_counts)
        print('-'*25)

    def update(self, context, arm, reward):
        # Update properly
        # These do satisfy the Robbins-Monro condition (provided exploration has a minimum rate
        # # and Every state-action pair is visited infinitely often
        # Option 1
        # n = self.arm_counts[context, arm]
        # q = self.q_table[context, arm]
        # self.q_table[context, arm] = q + (reward - q) / n
        # Option 2: Smoother
        td_target = reward
        td_error = td_target - self.q_table[context, arm]
        alpha = 1.0 / (1.0 + self.arm_counts[context, arm])
        self.q_table[context, arm] += alpha * td_error

        # Decay exploration
        self.explore_param = max(self.min_explore, self.explore_param * self.decay)

### Simulation driver

In [ ]:
def simulate(n_flags,num_agents,num_arms,iterations,explore_param,decay,min_explore,choice_type='egreedy'):
    # AGENT LIST num_arms, num_contexts, flag, explore_param=0.5, decay=0.99, min_explore=0.0001
    # Setup the list of agents.
  num_contexts = n_flags
  # This secures that flags are distributed as evenly as possible among agents
  agents_flags = [i % n_flags for i in range(num_agents)]
  random.shuffle(agents_flags)
  agent_list = []
  for x in range(num_agents):
      agent_list.append(ContextualBandit(num_arms, num_contexts,
                                       explore_param, decay,min_explore,
                                       flag=agents_flags[x],choice_type=choice_type))

  environment = BanditEnvironment(num_arms, num_contexts)
  act_d = {'cooperate':np.zeros(iterations),'defect':np.zeros(iterations)}
  flag_mean_rw_d = {flag: [] for flag in range(n_flags)}


  for t in tqdm(range(iterations)):
    # Track per-round payoffs safely (no shared list references)
    temppayoffs = defaultdict(list)

    # Randomize agent indexes and pair them up
    agents_indexes = list(range(num_agents))
    np.random.shuffle(agents_indexes)

    # If there's an odd number of agents, one is left unmatched
    unmatched_agent = None
    if len(agents_indexes) % 2 == 1:
        unmatched_agent = np.random.choice(agents_indexes)
        agents_indexes.remove(unmatched_agent)


    # Match agents in pairs
    for i in range(0, len(agents_indexes), 2):
        a = agents_indexes[i]
        b = agents_indexes[i + 1]

        # Identify agent flags (types)
        flag_a = agent_list[a].flag
        flag_b = agent_list[b].flag

        # Context for each agent is the other agent's flag
        a_context = flag_b
        b_context = flag_a

        # Agents choose actions based on the context
        a_action = agent_list[a].choice(a_context)
        b_action = agent_list[b].choice(b_context)

        # Translate actions and get rewards
        translated_a_action = environment.action_translation[a_action]
        translated_b_action = environment.action_translation[b_action]
        a_payoff, b_payoff = environment.get_reward_single_game(translated_a_action, translated_b_action)

        # Track payoffs by flag
        temppayoffs[flag_a].append(a_payoff)
        temppayoffs[flag_b].append(b_payoff)

        # Track action frequencies
        act_d[translated_a_action][t] += 1
        act_d[translated_b_action][t] += 1

        # Update agents based on feedback
        agent_list[a].update(a_context, a_action, a_payoff)
        agent_list[b].update(b_context, b_action, b_payoff)

    # agent_list[0].print_report()
    # Update running average payoff per flag
    for flag in range(n_flags):
        if temppayoffs[flag]:  # avoid empty list issue
            flag_mean_rw_d[flag].append(np.mean(temppayoffs[flag]))
        else:
            flag_mean_rw_d[flag].append(0.0)  # or np.nan if you'd rather leave it out

  return flag_mean_rw_d,act_d,agent_list

### Plotting helpers

In [ ]:
def plot_actions(act_d,n_flags):
  #show actions by round. Note that this is only showing the *primary* action, not the chosen partner action
#this is trying to get a sense of each agents choice, no double counting.
  xs = range(len(rew_d['defect']))
  plt.figure(figsize=(12,8))
  plt.plot(xs,rew_d['defect'],c='r',label='defect')
  plt.plot(xs,rew_d['cooperate'],c='b',label='cooperate')
  plt.xlabel('Round')
  plt.ylabel('Action Count')
  plt.suptitle(f"Action Counts at time steps with n_flags = {n_flags}")
  plt.legend()
  plt.show()

def plot_rewards(flag_mean_rw_d):
  for flag, rewards in flag_mean_rw_d.items():
      plt.plot(rewards, label=f"Flag {flag}")

  plt.title("Average Payoff Over Time by Flag")
  plt.xlabel("Iteration")
  plt.ylabel("Average Payoff")
  plt.legend()
  plt.grid(True)
  plt.show()

def plot_beliefs(n_flags,agent_list):
  # Randomly select one agent from each flag group
  for flag in range(n_flags):
      agents_with_flag = [agent for agent in agent_list if agent.flag == flag]
      if not agents_with_flag:
          continue  # skip if no agents with this flag
      example_agent = random.choice(agents_with_flag)
      q_values = example_agent.q_table  # shape: (num_contexts, num_arms)

      plt.figure()
      sns.heatmap(q_values, annot=True, cmap="viridis",
                  xticklabels=[f"Arm {i}" for i in range(num_arms)],
                  yticklabels=[f"Context {i}" for i in range(n_flags)])
      plt.title(f"Estimated Q-values for Random Agent in Flag {flag}")
      plt.xlabel("Action")
      plt.ylabel("Context (Other's Flag)")
      plt.tight_layout()
      plt.show()

def plot_ingroup_outgroup(iterations,agent_list):
  ingroup_payoffs = []
  outgroup_payoffs = []

  for t in range(iterations):
      ingroup_t = []
      outgroup_t = []
      for a in agent_list:
          flag = a.flag
          q_vals = a.q_table[flag]  # ingroup context
          ingroup_t.append(np.max(q_vals))  # best expected ingroup reward

          outgroup_contexts = [c for c in range(n_flags) if c != flag]
          outgroup_q = np.mean([np.max(a.q_table[c]) for c in outgroup_contexts])
          outgroup_t.append(outgroup_q)

      ingroup_payoffs.append(np.mean(ingroup_t))
      outgroup_payoffs.append(np.mean(outgroup_t))

  plt.plot(ingroup_payoffs, label="Ingroup Expected Reward")
  plt.plot(outgroup_payoffs, label="Outgroup Expected Reward")
  plt.title("Ingroup vs Outgroup Expectations Over Time")
  plt.xlabel("Iteration")
  plt.ylabel("Mean Max Expected Reward")
  plt.legend()
  plt.grid(True)
  plt.show()

### Run iteration 1

In [ ]:
# TRY THINGS OUT
iterations = 500  #iterations in simulation
num_agents = 500   #agents in population
explore_param=1 # Initial Exploratory parameter
decay=0.995
min_explore=0.0001
# Parameters
n_flags = 5
action_space=['cooperate','defect']
num_contexts = n_flags
num_arms = len(action_space)
hawk_dove_game = {'cooperate': {'cooperate':2,'defect':0},'defect': {'cooperate': 4,'defect':-2 }}
stag_hunt_game = {'cooperate': {'cooperate':3,'defect':0},'defect': {'cooperate': 1,'defect':1 }}


n_flags=1
flag_mean_rw_d,rew_d,agent_list = simulate(n_flags,num_agents,num_arms,iterations,
                                           explore_param,decay,min_explore,choice_type='ucb')
plot_actions(rew_d,n_flags)
plot_rewards(flag_mean_rw_d)
plot_beliefs(n_flags,agent_list)
plot_ingroup_outgroup(iterations,agent_list)

# for n_flags in range(1, 30, 5):
#   flag_mean_rw_d,rew_d,agent_list = simulate(n_flags,num_agents,num_arms,
#                                              iterations,explore_param,decay,min_explore,
#                                              choice_type='ucb')
#   plot_actions(rew_d,n_flags)
#   # plot_rewards(flag_mean_rw_d)

## Iteration 2 — Anonymous social norms on a grid (clustered flags)

Re-imports the same libraries (kept verbatim for fidelity to the original notebook progression).

In [ ]:
# Anonymous Social Norms on a Grid with Contextual Bandits

import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns

# --- GAME DEFINITION ---
stag_hunt_game = {
    'cooperate': {'cooperate': 3, 'defect': 0},
    'defect': {'cooperate': 1, 'defect': 1}
}

# --- GAME DEFINITION ---
prisoner_dilemma_game = {
    'cooperate': {'cooperate': 3, 'defect': 0},  # (C,C) = Reward (R), (C,D) = Sucker (S)
    'defect': {'cooperate': 4, 'defect': 1}   # (D,C) = Temptation (T), (D,D) = Punishment (P)
}

action_space = ['cooperate', 'defect']
n_flags = 5

### Environment and bandit (iteration 2)

In [ ]:
# --- ENVIRONMENT DEFINITION ---
class BanditEnvironment:
    def __init__(self, num_arms, num_contexts, game=prisoner_dilemma_game, n_flags=n_flags):
        self.game = game
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        self.timestep = 0
        self.action_translation = {0: 'cooperate', 1: 'defect'}

    def get_reward_single_game(self, p1_choice, p2_choice):
        self.timestep += 1
        p1_reward = self.game[p1_choice][p2_choice]
        p2_reward = self.game[p2_choice][p1_choice]
        return p1_reward, p2_reward

# --- CONTEXTUAL BANDIT AGENT ---
class ContextualBandit:
    def __init__(self, num_arms, num_contexts, explore_param, decay, min_explore, flag, choice_type='egreedy'):
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        self.q_table = np.zeros((num_contexts, num_arms))
        self.arm_counts = np.ones((num_contexts, num_arms))
        self.flag = flag
        self.choice_type = choice_type
        self.decay = decay
        self.min_explore = min_explore
        self.explore_param = explore_param
        self.time = 0
        self.last_reward = 0

    def choice(self, context):
        self.time += 1
        if self.choice_type == 'egreedy':
            if np.random.rand() < self.explore_param:
                choice = np.random.randint(self.num_arms)
            else:
                q_best = np.max(self.q_table[context])
                choice = np.random.choice(np.where(self.q_table[context] == q_best)[0])
        elif self.choice_type == 'ucb':
            UCB_estimation = self.q_table[context] + \
                self.explore_param * np.sqrt(np.log(self.time + 1) / (self.arm_counts[context] + 1e-5))
            choice = np.argmax(UCB_estimation)
        elif self.choice_type == 'softmax':
            q_values = self.q_table[context]
            exp_q = np.exp((q_values - np.max(q_values)) / self.explore_param)
            probs = exp_q / np.sum(exp_q)
            choice = np.random.choice(self.num_arms, p=probs)

        self.arm_counts[context, choice] += 1
        return choice

    def update(self, context, arm, reward):
        td_target = reward
        td_error = td_target - self.q_table[context, arm]
        alpha = 1.0 / (1.0 + self.arm_counts[context, arm])
        self.q_table[context, arm] += alpha * td_error
        self.explore_param = max(self.min_explore, self.explore_param * self.decay)
        self.last_reward = reward

### GridWorld with clustered flag positions (iteration 2)

In [ ]:
# --- GRID IMPLEMENTATION ---
class GridWorld:
    def __init__(self, size, num_agents, env, **bandit_kwargs):
        self.size = size
        self.grid = [[[] for _ in range(size)] for _ in range(size)]
        self.env = env
        self.agents = []

        # Randomly assign starting positions per flag
        flag_positions = self._generate_flag_positions(n_flags)
        agents_per_flag = num_agents // n_flags

        for flag in range(n_flags):
            start_x, start_y = flag_positions[flag]
            for _ in range(agents_per_flag):
                agent = ContextualBandit(env.num_arms, env.num_contexts, flag=flag, **bandit_kwargs)
                self.agents.append((agent, start_x, start_y))
                self.grid[start_x][start_y].append(agent)

    def _generate_flag_positions(self, n):
        positions = set()
        while len(positions) < n:
            pos = (np.random.randint(0, self.size), np.random.randint(0, self.size))
            positions.add(pos)
        return list(positions)

    def move_agents(self):
        self.grid = [[[] for _ in range(self.size)] for _ in range(self.size)]
        new_positions = []
        for agent, x, y in self.agents:
            dx, dy = random.choice([(0,1), (0,-1), (1,0), (-1,0)])
            nx, ny = (x + dx) % self.size, (y + dy) % self.size
            new_positions.append((agent, nx, ny))
            self.grid[nx][ny].append(agent)
        self.agents = new_positions

    def step(self):
        self.move_agents()
        for agent, x, y in self.agents:
            local_agents = self.grid[x][y][:]  # agents in the same cell
            local_agents = [a for a in local_agents if a is not agent]
            if local_agents:
                partner = random.choice(local_agents)
                self.play_anonymous_game(agent, partner)

    def play_anonymous_game(self, a1, a2):
        context1, context2 = a1.flag, a2.flag
        choice1 = a1.choice(context1)
        choice2 = a2.choice(context2)
        act1 = self.env.action_translation[choice1]
        act2 = self.env.action_translation[choice2]
        r1, r2 = self.env.get_reward_single_game(act1, act2)
        a1.update(context1, choice1, r1)
        a2.update(context2, choice2, r2)
        return act1, act2, a1.flag, r1, a2.flag, r2

    def visualize(self):
        display = np.zeros((self.size, self.size))
        for agent, x, y in self.agents:
            top_choice = np.argmax(agent.q_table[agent.flag])
            display[x, y] = 1 if top_choice == 0 else -1
        plt.imshow(display, cmap='bwr', vmin=-1, vmax=1)
        plt.title('Blue = Cooperate, Red = Defect')
        plt.colorbar()
        plt.show()

### Run iteration 2

In [ ]:
# --- SIMULATION ---
num_agents = 500
grid_size = 50
iterations = 100
n_flags = 1
bandit_env = BanditEnvironment(num_arms=2, num_contexts=n_flags)
grid_sim = GridWorld(size=grid_size, num_agents=num_agents, env=bandit_env,
                     explore_param=0.5, decay=0.99, min_explore=0.01, choice_type='ucb')

act_d = {'cooperate': np.zeros(iterations), 'defect': np.zeros(iterations)}
flag_mean_rw_d = {flag: [] for flag in range(n_flags)}

for t in range(iterations):
    grid_sim.move_agents()
    coop_count, defect_count = 0, 0
    flag_rewards = defaultdict(list)

    for agent, x, y in grid_sim.agents:
        local_agents = grid_sim.grid[x][y][:]
        local_agents = [a for a in local_agents if a is not agent]
        if local_agents:
            partner = random.choice(local_agents)
            act1, act2, f1, r1, f2, r2 = grid_sim.play_anonymous_game(agent, partner)
            coop_count += (act1 == 'cooperate') + (act2 == 'cooperate')
            defect_count += (act1 == 'defect') + (act2 == 'defect')
            flag_rewards[f1].append(r1)
            flag_rewards[f2].append(r2)

    act_d['cooperate'][t] = coop_count
    act_d['defect'][t] = defect_count

    for flag in range(n_flags):
        mean_rw = np.mean(flag_rewards[flag]) if flag_rewards[flag] else 0
        flag_mean_rw_d[flag].append(mean_rw)

    if t % 20 == 0:
        print(f"Timestep {t}")
        grid_sim.visualize()

# Plotting
plt.figure(figsize=(10,4))
plt.plot(act_d['cooperate'], label='Cooperate')
plt.plot(act_d['defect'], label='Defect')
plt.legend()
plt.title("Number of Cooperative vs Defective Actions")
plt.xlabel("Timestep")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10,4))
for flag in range(n_flags):
    plt.plot(flag_mean_rw_d[flag], label=f"Flag {flag}")
plt.legend()
plt.title("Average Reward per Flag Group")
plt.xlabel("Timestep")
plt.ylabel("Average Reward")
plt.show()

## Iteration 3 — Random flag positions

Re-imports the libraries again, like the original notebook. Iteration 3 differs from iteration 2 in that agents start at random positions (not clustered per flag) and use Moore-neighbourhood neighbours instead of same-cell partners.

In [ ]:
# Anonymous Social Norms on a Grid with Contextual Bandits

import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns

# --- GAME DEFINITION ---
stag_hunt_game = {
    'cooperate': {'cooperate': 3, 'defect': 0},
    'defect': {'cooperate': 1, 'defect': 1}
}

action_space = ['cooperate', 'defect']
n_flags = 5

### Environment and bandit (iteration 3)

In [ ]:
# --- ENVIRONMENT DEFINITION ---
class BanditEnvironment:
    def __init__(self, num_arms, num_contexts, game=prisoner_dilemma_game, n_flags=n_flags):
        self.game = game
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        self.timestep = 0
        self.action_translation = {0: 'cooperate', 1: 'defect'}

    def get_reward_single_game(self, p1_choice, p2_choice):
        self.timestep += 1
        p1_reward = self.game[p1_choice][p2_choice]
        p2_reward = self.game[p2_choice][p1_choice]
        return p1_reward, p2_reward

# --- CONTEXTUAL BANDIT AGENT ---
class ContextualBandit:
    def __init__(self, num_arms, num_contexts, explore_param, decay, min_explore, flag, choice_type='egreedy'):
        self.num_arms = num_arms
        self.num_contexts = num_contexts
        self.q_table = np.zeros((num_contexts, num_arms))
        self.arm_counts = np.ones((num_contexts, num_arms))
        self.flag = flag
        self.choice_type = choice_type
        self.decay = decay
        self.min_explore = min_explore
        self.explore_param = explore_param
        self.time = 0

    def choice(self, context):
        self.time += 1
        if self.choice_type == 'egreedy':
            if np.random.rand() < self.explore_param:
                choice = np.random.randint(self.num_arms)
            else:
                q_best = np.max(self.q_table[context])
                choice = np.random.choice(np.where(self.q_table[context] == q_best)[0])
        elif self.choice_type == 'ucb':
            UCB_estimation = self.q_table[context] + \
                self.explore_param * np.sqrt(np.log(self.time + 1) / (self.arm_counts[context] + 1e-5))
            print(self.q_table[context])
            print(UCB_estimation)
            choice = np.argmax(UCB_estimation)
        elif self.choice_type == 'softmax':
            q_values = self.q_table[context]
            exp_q = np.exp((q_values - np.max(q_values)) / self.explore_param)
            probs = exp_q / np.sum(exp_q)
            choice = np.random.choice(self.num_arms, p=probs)

        self.arm_counts[context, choice] += 1
        return choice

    def update(self, context, arm, reward):
        td_target = reward
        td_error = td_target - self.q_table[context, arm]
        alpha = 1.0 / (1.0 + self.arm_counts[context, arm])
        self.q_table[context, arm] += alpha * td_error
        self.explore_param = max(self.min_explore, self.explore_param * self.decay)

### GridWorld with random positions and neighbours (iteration 3)

In [ ]:
# --- GRID IMPLEMENTATION ---
class GridWorld:
    def __init__(self, size, num_agents, env, **bandit_kwargs):
        self.size = size
        self.grid = [[[] for _ in range(size)] for _ in range(size)]
        self.env = env
        self.agents = []
        for _ in range(num_agents):
            x, y = np.random.randint(0, size), np.random.randint(0, size)
            flag = np.random.randint(0, n_flags)
            agent = ContextualBandit(env.num_arms, env.num_contexts, flag=flag, **bandit_kwargs)
            self.agents.append((agent, x, y))
            self.grid[x][y].append(agent)

    def move_agents(self):
        self.grid = [[[] for _ in range(self.size)] for _ in range(self.size)]
        new_positions = []
        for agent, x, y in self.agents:
            dx, dy = random.choice([(0,1), (0,-1), (1,0), (-1,0)])
            nx, ny = (x + dx) % self.size, (y + dy) % self.size
            new_positions.append((agent, nx, ny))
            self.grid[nx][ny].append(agent)
        self.agents = new_positions

    def step(self):
        self.move_agents()
        for agent, x, y in self.agents:
            neighbors = self.get_neighbors(x, y)
            if neighbors:
                partner = random.choice(neighbors)
                self.play_anonymous_game(agent, partner)

    def get_neighbors(self, x, y):
        neighbors = []
        for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
            nx, ny = (x + dx) % self.size, (y + dy) % self.size
            neighbors.extend(self.grid[nx][ny])
        return neighbors

    def play_anonymous_game(self, a1, a2):
        context1, context2 = a1.flag, a2.flag
        choice1 = a1.choice(context1)
        choice2 = a2.choice(context2)
        act1 = self.env.action_translation[choice1]
        act2 = self.env.action_translation[choice2]
        r1, r2 = self.env.get_reward_single_game(act1, act2)
        a1.update(context1, choice1, r1)
        a2.update(context2, choice2, r2)

    def visualize(self):
        display = np.zeros((self.size, self.size))
        for agent, x, y in self.agents:
            top_choice = np.argmax(agent.q_table[agent.flag])
            display[x, y] = 1 if top_choice == 0 else -1
        plt.imshow(display, cmap='bwr', vmin=-1, vmax=1)
        plt.title('Blue = Cooperate, Red = Defect')
        plt.colorbar()
        plt.show()

### Run iteration 3

In [ ]:
# --- SIMULATION ---
num_agents = 100
grid_size = 15
bandit_env = BanditEnvironment(num_arms=2, num_contexts=n_flags)
grid_sim = GridWorld(size=grid_size, num_agents=num_agents, env=bandit_env,
                     explore_param=0.5, decay=0.99, min_explore=0.01, choice_type='egreedy')

for t in range(100):
    grid_sim.step()
    if t % 20 == 0:
        print(f"Timestep {t}")
        grid_sim.visualize()